In [1]:
import json
import pandas as pd
import os
import time
import yt_dlp
from yt_dlp.utils import download_range_func # <-- THÊM DÒNG IMPORT NÀY

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN VÀ TỪ VỰNG
# ==========================================
# (Giả sử bạn đang chạy code từ thư mục notebooks/)
JSON_PATH = '../MSASL_train.json'
OUTPUT_DIR = '../data/raw_videos/'
VALID_CSV_PATH = '../data/MSASL_train_valid_only.csv'

# Lọc ra 5 từ vựng cơ bản để làm demo project trước
TARGET_WORDS = ['hello', 'thank you', 'please', 'help', 'sorry']

# ==========================================
# 1. HÀM TẢI VÀ CẮT VIDEO BẰNG YT-DLP
# ==========================================
def download_video_segment(url, start_time, end_time, save_path):
    """
    Tải một đoạn video từ YouTube dựa trên thời gian bắt đầu và kết thúc.
    Trả về trạng thái tải (success, private, deleted, rate_limited, error).
    """
    ydl_opts = {
        # THAY ĐỔI 1: Ưu tiên tải độ phân giải tối thiểu 720p, không ép đuôi MP4 nữa
        'format': 'bestvideo[height>=720]/bestvideo/best',
        
        # Đổi tên file lưu trữ bỏ phần đuôi .mp4 cứng, thay bằng %(ext)s để yt-dlp tự nhận diện
        'outtmpl': save_path.replace('.mp4', '.%(ext)s'), 
        
        'download_ranges': download_range_func(None, [(float(start_time), float(end_time))]),
        'force_keyframes_at_cuts': True, 
        
        # THAY ĐỔI 2: Ép FFmpeg sử dụng thông số nén Lossless (Không mất dữ liệu)
        # -crf 18: Tham số Constant Rate Factor. Số càng nhỏ càng nét (18 là gần như độ nét gốc).
        # -preset fast: Tăng tốc độ cắt video mà không làm giảm chất lượng.
        'external_downloader_args': {
            'ffmpeg_o': ['-crf', '18', '-preset', 'fast']
        },
        
        'quiet': True,
        'no_warnings': True,
        'ignoreerrors': False,
        'sleep_interval': 3,
        'max_sleep_interval': 8,
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
        }
    }
    
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        return "success"
        
    except yt_dlp.utils.DownloadError as e:
        error_msg = str(e).lower()
        if "private video" in error_msg:
            return "private"
        elif "unavailable" in error_msg or "removed" in error_msg or "terminated" in error_msg:
            return "deleted"
        elif "http error 429" in error_msg:
            return "rate_limited"
        else:
            return f"error: {error_msg}"
            
    except Exception as e:
        return f"unknown_error: {str(e)}"

# ==========================================
# 2. ĐỌC DỮ LIỆU VÀ CHUẨN BỊ THƯ MỤC
# ==========================================
print("Đang đọc file JSON...")
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
filtered_df = df[df['clean_text'].isin(TARGET_WORDS)].copy()

print(f"Tổng số video gốc: {len(df)}")
print(f"Số lượng video cần xử lý cho {len(TARGET_WORDS)} từ vựng: {len(filtered_df)}\n")

# Tạo thư mục con cho từng từ vựng
for word in TARGET_WORDS:
    os.makedirs(os.path.join(OUTPUT_DIR, word), exist_ok=True)

# ==========================================
# 3. VÒNG LẶP XỬ LÝ VÀ TẢI VIDEO
# ==========================================
successful_downloads = []
dead_links_count = 0

print("-" * 50)
print(f"Bắt đầu xử lý {len(filtered_df)} video...")
print("-" * 50)

for index, row in filtered_df.iterrows():
    word = row['clean_text']
    url = row['url']
    start_time = row['start_time']
    end_time = row['end_time']
    signer_id = row['signer_id']
    
    # Định dạng tên file: từ_vựng_idsign_index.mp4
    filename = f"{word}_{signer_id}_{index}.mp4"
    save_path = os.path.join(OUTPUT_DIR, word, filename)
    
    # Nếu file đã tải rồi thì bỏ qua
    if os.path.exists(save_path):
        print(f"[SKIPPED] Đã có sẵn: {filename}")
        successful_downloads.append(row)
        continue

    print(f"[DOWNLOADING] {word} | Link: {url}")
    status = download_video_segment(url, start_time, end_time, save_path)
    
    if status == "success":
        print("  -> Thành công!")
        successful_downloads.append(row)
        
    elif status == "rate_limited":
        print("  -> [CẢNH BÁO] Bị YouTube giới hạn tốc độ (Lỗi 429).")
        print("  -> Đang tạm dừng hệ thống 2 phút để tránh bị khóa IP...")
        time.sleep(120) # Ngủ 120 giây rồi tiếp tục vòng lặp
        
    elif status == "private":
        print("  -> Thất bại: Video Private. Bỏ qua.")
        dead_links_count += 1
        
    elif status == "deleted":
        print("  -> Thất bại: Video đã bị xóa. Bỏ qua.")
        dead_links_count += 1
        
    else:
        print(f"  -> Thất bại do lỗi: {status}")
        dead_links_count += 1

# ==========================================
# 4. LƯU KẾT QUẢ DATA SẠCH
# ==========================================
if successful_downloads:
    clean_df = pd.DataFrame(successful_downloads)
    # Lưu ra file CSV để giai đoạn sau (MediaPipe) chỉ đọc file này
    clean_df.to_csv(VALID_CSV_PATH, index=False)

print("-" * 50)
print("BÁO CÁO TỔNG KẾT:")
print(f"- Tổng số link đã duyệt: {len(filtered_df)}")
print(f"- Tải thành công / Đã có sẵn: {len(successful_downloads)}")
print(f"- Link chết/Lỗi (đã loại bỏ): {dead_links_count}")
print(f"- Đã lưu danh sách các video hợp lệ ra file: {VALID_CSV_PATH}")
print("-" * 50)

Đang đọc file JSON...
Tổng số video gốc: 16054
Số lượng video cần xử lý cho 5 từ vựng: 168

--------------------------------------------------
Bắt đầu xử lý 168 video...
--------------------------------------------------
[DOWNLOADING] help | Link: https://www.youtube.com/watch?v=htsdwxJ-fTo


ERROR: [youtube] htsdwxJ-fTo: This video is not available


  -> Thất bại do lỗi: error: error: [youtube] htsdwxj-fto: this video is not available
[DOWNLOADING] help | Link: www.youtube.com/watch?v=3DbWOEtUigU


ERROR: [youtube] 3DbWOEtUigU: Video unavailable


  -> Thất bại: Video đã bị xóa. Bỏ qua.
[DOWNLOADING] sorry | Link: www.youtube.com/watch?v=zCUZOnKoKWQ


...

  -> Thất bại: Video đã bị xóa. Bỏ qua.
[DOWNLOADING] please | Link: www.youtube.com/watch?v=wNS3kifA8-Y


ERROR: [youtube] wNS3kifA8-Y: Video unavailable


  -> Thất bại: Video đã bị xóa. Bỏ qua.
[DOWNLOADING] thank you | Link: https://www.youtube.com/watch?v=2zIXu4qXXlk
  -> Thành công!                                        
[DOWNLOADING] help | Link: https://www.youtube.com/watch?v=TPmpYP8l888&t=528s


ERROR: [youtube] TPmpYP8l888: Private video. Sign in if you've been granted access to this video. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


  -> Thất bại: Video Private. Bỏ qua.
--------------------------------------------------
BÁO CÁO TỔNG KẾT:
- Tổng số link đã duyệt: 168
- Tải thành công / Đã có sẵn: 104
- Link chết/Lỗi (đã loại bỏ): 64
- Đã lưu danh sách các video hợp lệ ra file: ../data/MSASL_train_valid_only.csv
--------------------------------------------------
